In [ ]:
from itertools import chain
import networkx as nx
import matplotlib.pyplot as plt
from collections import Counter
import numpy as np
import sys

ITERATIONS = 1000
RULE = ([[1,2,1],[2,3,4]],[[5,4,5],[5,3,1],[2,4,1]])
graph = [[0,0,0],[0,0,0]]
'''
RULE = ([[1,2,2],[3,1,4]],[[2,5,2],[2,3,5],[4,5,5]])
graph = [[0,0,0],[0,0,0]]
'''
timesteps = {tuple(edge):0 for edge in graph}
end_steps = 300
next_index = max(chain.from_iterable(graph))+1
SHOW_EVERY = 1000

def show(graph):
    split = list(chain.from_iterable([(edge[i], edge[i+1]) for i in range(len(edge)-1)] for edge in graph))
    frequency = Counter(split)
    weights = {edge:(count*(max(frequency.values()))**(-1/2)/20) for edge, count in frequency.items()}
    G = nx.MultiGraph()
    G.add_edges_from([(u, v, {'weight': weights[(u, v)]}) for u, v in split])
    pos = nx.kamada_kawai_layout(G)
    #pos = nx.spring_layout(G)
    #pos = nx.planar_layout(G)
    width = [weights.get((u, v), 0.1) * 10 for u, v in G.edges()]
    nx.draw(G,pos,node_size=1e3/len(list(chain.from_iterable(split))), arrowsize=5e1/(len(list(chain.from_iterable(split))))**(1/4), width=width)
    plt.show()

def pattern(rule,list,map):
    pattern = []
    for i,value in enumerate(rule):
        if(value not in map):
            map[value] = list[i]
        pattern.append(map[value])
    return pattern

def rule_check(graph,used_i,map={},rule_index=0,indices=[],first=True):
    if(not map):
        map = {}
    if(not indices):
        indices = []
    for i in range(len(graph)):
        old_map = map.copy()
        local_indices = indices.copy()
        if(graph[i]==pattern(RULE[0][rule_index],graph[i],map))and(i not in used_i):
            local_indices.append(i)
            used_i.append(i)
            if(rule_index+1==len(RULE[0])):
                return local_indices,used_i,map,first
            else:
                n1,n2,n3,new = rule_check(graph,used_i,map,rule_index+1,local_indices,False)
                if(new):
                    map = old_map
                    continue
                else:
                    return n1,n2,n3,new
        else:
            map = old_map
    else:
        return [],[],map,True


print('Initial Graph')
show(graph)
for step in range(ITERATIONS):
    indices,used_i,map,first = rule_check(graph,used_i=[],indices=[],map={})
    save_map = [map.copy()]
    updates = [indices.copy()]
    while(indices):
        n1,n2,n3,first = rule_check(graph,used_i,indices=[],map={})
        if(not n1):
            break
        indices,used_i,map = n1,n2,n3
        save_map.append(map)
        updates.append(indices)
    if(not used_i):
        sys.exit('No Matches Found on Step: ' + str(step))
    for u in sorted(np.ndarray.flatten(np.array(updates)),reverse=True):
        graph.pop(u)
    for i in range(len(updates)):
        for edge in RULE[1]:
            temp = []
            for e in edge:
                if(e not in save_map[i]):
                    save_map[i][e] = next_index
                    next_index += 1
                temp.append(save_map[i][e])
            graph.append(temp)
            timesteps[tuple(temp)] = step
    if(((step+1)%SHOW_EVERY==0)or((step+1)<=0)):
        print('_'*100+'\n\nStep: '+str(step+1))
        show(graph)
for i in range(ITERATIONS-100,1,-100):
    new_graph = []
    for edge in graph:
        if(timesteps[tuple(edge)]<=i):
            new_graph.append(edge)
    print('_'*100+'\n\nFinal Graph before '+str(i)+' steps')
    show(new_graph)
    new_graph = []
    for edge in graph:
        if(timesteps[tuple(edge)]>=i):
            new_graph.append(edge)
    print('\n\nFinal Graph after '+str(i)+' steps')
    show(new_graph)

In [ ]:
from itertools import chain
import networkx as nx
import matplotlib.pyplot as plt
from collections import Counter
import numpy as np
import sys

ITERATIONS = 12
RULE = ([[1,2],[1,3]],[[1,2],[1,4],[2,4],[3,4]])
graph = [[1, 2], [2, 3], [3, 4], [2, 4]]
timesteps = {tuple(edge):0 for edge in graph}
next_index = max(chain.from_iterable(graph))+1
SHOW_EVERY = 1

def show(graph):
    split = list(chain.from_iterable([(edge[i], edge[i+1]) for i in range(len(edge)-1)] for edge in graph))
    frequency = Counter(split)
    weights = {edge:(count*(max(frequency.values()))**(-1/2)/20) for edge, count in frequency.items()}
    G = nx.MultiGraph()
    G.add_edges_from([(u, v, {'weight': weights[(u, v)]}) for u, v in split])
    pos = nx.kamada_kawai_layout(G)
    #pos = nx.spring_layout(G)
    #pos = nx.planar_layout(G)
    width = [weights.get((u, v), 0.1) * 10 for u, v in G.edges()]
    nx.draw(G,pos,node_size=1e3/(len(graph)+1), arrowsize=5e1/(len(graph)+1)**(1/4), width=width)
    plt.show()

def rule_check(graph,used_i,map={},rule_index=0,indices=[],first=True):
    map,indices = map or {}, indices or []
    for i,edge in enumerate(graph):
        old_map,local_indices = map.copy(),indices+[i]
        if(edge==[map.setdefault(value,edge[j]) for j,value in enumerate(RULE[0][rule_index])])and(i not in used_i):
            if(rule_index+1==len(RULE[0])):
                return local_indices,used_i+[i],map,first
            result = rule_check(graph,used_i+[i],map,rule_index+1,local_indices,False)
            if(not result[3]):
                return result
        map = old_map
    return [],[],map,True

print('Initial Graph')
show(graph)
for step in range(ITERATIONS):
    indices,used_i,map,first = rule_check(graph,used_i=[],indices=[],map={})
    save_map = [map.copy()]
    updates = [indices.copy()]
    while(True):
        n1,n2,n3,first = rule_check(graph,used_i,indices=[],map={})
        if(not n1):
            break
        indices,used_i,map = n1,n2,n3
        save_map.append(map)
        updates.append(indices)
    if(not used_i):
        sys.exit('No Matches Found on Step: '+str(step))
    for i in sorted(np.ndarray.flatten(np.array(updates)),reverse=True):
        graph.pop(i)
    for i in range(len(updates)):
        for edge in RULE[1]:
            temp = []
            for e in edge:
                if(e not in save_map[i]):
                    save_map[i][e] = next_index
                    next_index += 1
                temp.append(save_map[i][e])
            graph.append(temp)
            timesteps[tuple(temp)] = step
    if(((step+1)%SHOW_EVERY==0)or((step+1)<=1)):
        print('_'*100+'\n\nStep: '+str(step+1))
        show(graph)

for i in range(ITERATIONS,0,-1):
    print('_'*100+'\n\nFinal Graph before '+str(i)+' steps')
    show([edge for edge in graph if timesteps[tuple(edge)]<=i])
    print('\n\nFinal Graph after '+str(i)+' steps')
    show([edge for edge in graph if timesteps[tuple(edge)]>=i])